In [ ]:

import pandas as pd
import numpy as np
import random
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import KFold
from sklearn.svm import SVC
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    confusion_matrix,
)
from sklearn.calibration import calibration_curve

# Fixing all random seeds
SEED = 2025
np.random.seed(SEED)
random.seed(SEED)


nb_classes = 2
labels = ["Pas de Récidive", "Récidive"]
ticksx = np.arange(nb_classes) + 0.5
ticksy = np.arange(nb_classes) + 0.5

## utils 

In [ ]:

def compute_metrics_fm(y_test, y_pred,pids):
    new_y_test = []
    new_y_pred = []
    # copute median pred on a patient
    for patient_id in set(pids):
        indexes = [i for i,pid in enumerate(pids) if pid ==patient_id]
        if len(indexes)>1:
            median_pred = np.median([y_pred[i] for i in indexes])
            new_y_pred.append(median_pred)
        else:
            new_y_pred.append(y_pred[indexes[0]])
        new_y_test.append(y_test[indexes[0]])
    new_y_pred = np.array(new_y_pred)
    new_y_test = np.array(new_y_test)
    accuracy = accuracy_score(new_y_test, new_y_pred>0.5)
    f1 = f1_score(new_y_test, new_y_pred>0.5)
    PPV = precision_score(new_y_test, new_y_pred>0.5)
    cm = confusion_matrix(new_y_test, new_y_pred>0.5)
    tn, fp, fn, tp = cm.ravel().tolist()
    NPV = tn/(tn+fn) if (tn+fn)>0 else 0
    Sensitivity = recall_score(new_y_test, new_y_pred>0.5)
    Specificity = tn / (tn + fp) if (tn+fp)>0 else 0 
    fpr, tpr, thresholds = roc_curve(new_y_test, new_y_pred)
    roc_auc = auc(fpr, tpr)
    return accuracy, f1, PPV,NPV, Sensitivity,Specificity,fpr,tpr,roc_auc, new_y_pred,new_y_test


def getWrongPatients(yhat, ytrue, patients):
    if len(np.where(yhat != ytrue)) != 0:
        return patients[np.where(yhat != ytrue)]
    else:
        print("No Misclassified Patients.")
        return np.array([0])

In [ ]:
import torch
import os
device = 'cpu'

class EmbedingDataset(torch.utils.data.Dataset):
    def __init__(self, embeddings_path, labels_path,split ='train'):
        if split == 'train':
            df0 = pd.read_excel(labels_path, sheet_name="PB").dropna(subset=["Patient"])
            self.dict_labels = {int(df0.at[i, "Patient"]): int(df0.at[i,"Récidive avant 2 ans"]) for i in range(len(df0))}
        elif split == 'test_HM':
            df1 = pd.read_excel(labels_path, sheet_name="HMN").dropna(subset=["Patient"])
            self.dict_labels = {int(df1.at[i, "Patient"]): int(df1.at[i,"Récidive avant 2 ans"]) for i in range(len(df1))}
        elif split == 'test_BJN':
            df2 = pd.read_excel(labels_path, sheet_name="BJN").dropna(subset=["Patient"])
            self.dict_labels = {int(df2.at[i, "Patient"]): int(df2.at[i,"Récidive avant 2 ans"]) for i in range(len(df2))}
        self.pt_files = {f.split('_')[0] : os.path.join(embeddings_path, f) for f in os.listdir(embeddings_path) if f.endswith('_features.pt')}
        # filter pt_files to keep only those in dict_labels
        self.pt_files = {k: v for k, v in self.pt_files.items() if int(k[:-1]) in self.dict_labels}
        self.slide_ids = list(self.pt_files.keys())
        
    def __len__(self):
        return len(self.pt_files)

    def __getitem__(self, idx):
        slide_id = self.slide_ids[idx]
        embedding = torch.load(self.pt_files[slide_id])['last_layer_embed']
        label = self.dict_labels[int(slide_id[:-1])]
        target = torch.tensor(label, dtype=torch.float32)
        patient_id = slide_id[:-1]
        return embedding.to(device), target.to(device),patient_id
    
trainDataset = EmbedingDataset(embeddings_path='../data/features',labels_path='../data/Label_slides.xlsx',split='train')
list_tensors = [data.squeeze() for data,_,_ in trainDataset]
df_kb_fm = pd.DataFrame(list_tensors)
df_kb_fm["Récidive Globale"] = [int(label) for _,label,_ in trainDataset]
df_kb_fm["patient"] = [int(pid) for _,_,pid in trainDataset]

list_tensors=[]
list_label = []
list_pids = []
for data,label,pid in trainDataset:
    list_tensors.append(data.squeeze())
    list_label.append(int(label))
    list_pids.append(int(pid))

list_tensors = [data.squeeze() for data,_,_ in trainDataset]
df_kb_fm = pd.DataFrame(list_tensors)
df_kb_fm["Récidive Globale"] = list_label
df_kb_fm["patient"] = list_pids


HM_Dataset = EmbedingDataset(embeddings_path='../data/features',labels_path='../data/Label_slides.xlsx',split='test_HM')

list_tensors=[]
list_label = []
list_pids = []
for data,label,pid in HM_Dataset:
    list_tensors.append(data.squeeze())
    list_label.append(int(label))
    list_pids.append(int(pid))

list_tensors = [data.squeeze() for data,_,_ in HM_Dataset]
df_hm_fm = pd.DataFrame(list_tensors)
df_hm_fm["Récidive Globale"] = list_label
df_hm_fm["patient"] = list_pids

BJ_Dataset = EmbedingDataset(embeddings_path='../data/features',labels_path='../data/Label_slides.xlsx',split='test_BJN')

list_tensors=[]
list_label = []
list_pids = []
for data,label,pid in BJ_Dataset:
    list_tensors.append(data.squeeze())
    list_label.append(int(label))
    list_pids.append(int(pid))

list_tensors = [data.squeeze() for data,_,_ in BJ_Dataset]
df_bj_fm = pd.DataFrame(list_tensors)
df_bj_fm["Récidive Globale"] = list_label
df_bj_fm["patient"] = list_pids

cols_to_scale = np.arange(768)
FINAL_COLS = cols_to_scale

## 5-folds cross validation

In [ ]:
X = df_kb_fm[["patient"] + list(FINAL_COLS)]
y = df_kb_fm[["patient", "Récidive Globale"]]
patients = df_kb_fm["patient"].values

In [ ]:
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from xgboost import XGBClassifier
# create a list of models 
models = []
# try a gaussian svm
svm  = make_pipeline(RobustScaler(), SVC(probability=True))
models.append(svm)
# try a random forest
rf = make_pipeline(RobustScaler(), RandomForestClassifier()) 
models.append(rf)
# try a gradient boosting
gb = make_pipeline(RobustScaler(), GradientBoostingClassifier()) 
models.append(gb)
# try a adaboost
ada = make_pipeline(RobustScaler(), AdaBoostClassifier() )
models.append(ada)
# try a xgboost
xgb = make_pipeline(RobustScaler(), XGBClassifier())
models.append(xgb)
# try a decision tree
dt = make_pipeline(RobustScaler(), DecisionTreeClassifier() )
models.append(dt)
# try a mlp
mlp = make_pipeline(RobustScaler(),MLPClassifier(hidden_layer_sizes=(), activation='identity') )
models.append(mlp)


In [ ]:

result_dict = {}
n_splits = 5

patients_set = list(set(X["patient"]))
labels = [ y[y.patient == p].iloc[0].at["Récidive Globale"] for p in patients_set]

sss = KFold(n_splits=n_splits)
for model in models:
  scores, f1s, ppvs,npvs,senss,specs,fprs,tprs,rocs = [], [],[], [],[], [],[], [],[]
  for split_id, (train_idx, test_idx) in enumerate(
    sss.split(patients_set,labels), 1
):
    train_patients = sorted([patients_set[idx] for idx in train_idx])
    test_patients = sorted([patients_set[idx] for idx in test_idx])

    X_train, X_test = (
        X.loc[X["patient"].isin(train_patients)],
        X.loc[X["patient"].isin(test_patients)],
    )
    X_train, X_test = X_train[FINAL_COLS], X_test[FINAL_COLS]

    y_train, y_test = (
        y.loc[y["patient"].isin(train_patients)]["Récidive Globale"].values,
        y.loc[y["patient"].isin(test_patients)]["Récidive Globale"].values,
    )

    model.fit(X_train, y_train)
    y_pred = model.predict_proba(X_test)[:,1]

    acc, f1, PPV,NPV, Sensitivity,Specificity,fpr,tpr,roc_auc, new_y_pred,new_y_test = compute_metrics_fm(y_test, y_pred,test_patients)
    
    ppvs.append(PPV)
    npvs.append(NPV)
    senss.append(Sensitivity)
    specs.append(Specificity)
    fprs.append(fpr)
    tprs.append(tpr)
    rocs.append(roc_auc)
    scores.append(acc)
    f1s.append(f1)
    print(f"Split {split_id}, accuracy = {acc:.3f}, f1-score = {f1:.3f}")
  result_dict[model]= {'accuracy' : scores, 'f1' : f1s,'PPV': ppvs, 'NPV':npvs,
                       'Sensitivity':senss,'Specificity':specs,'FPR':fprs,'TPR':tprs
                       ,'auc':rocs}

In [ ]:
for model in result_dict.keys():
    results = result_dict[model]
    plt.figure()
    x = ['accuracy','f1','Sensitivity','Specificity','PPV','NPV']
    to_plot = [results['accuracy'],results['f1'],results['Sensitivity'],results['Specificity'],results['PPV'],results['NPV']]
    y = [np.mean(res_list) for res_list in to_plot]
    err = [np.std(res_list) for res_list in to_plot]
    rects = plt.bar(x,y, yerr=err)
    plt.bar_label(rects,labels= [f'{y[i]:.2f}±{err[i]:.2f}' for i in range(len(y))])
    plt.xlabel('Metrics')
    plt.ylabel('Performances')
    plt.title(model.steps[1][0])
    plt.show()

## Final training

In [ ]:

models = {
    "GradientBoosting": GradientBoostingClassifier()
    
}

In [ ]:
from sklearn.metrics import precision_recall_curve
cols_to_scale = np.arange(768)

train_fm = df_kb_fm.copy()

robust_scaler = RobustScaler()
data_scaled = robust_scaler.fit_transform(train_fm[cols_to_scale])
# Kremlin Bicetre
train_fm[cols_to_scale] = data_scaled

# HenriMondor
data_scaled = robust_scaler.transform(df_hm_fm[cols_to_scale])
df_hm_fm_scaled = df_hm_fm.copy()
df_hm_fm_scaled[cols_to_scale] = data_scaled

# Beaujon
data_scaled = robust_scaler.transform(df_bj_fm[cols_to_scale])
df_bj_fm_scaled = df_bj_fm.copy()
df_bj_fm_scaled[cols_to_scale] = data_scaled

FINAL_COLS = cols_to_scale
X_train = train_fm[FINAL_COLS]
y_train = train_fm["Récidive Globale"]

patients_train = train_fm["patient"]

X_hm = df_hm_fm_scaled[FINAL_COLS]
y_hm = df_hm_fm_scaled["Récidive Globale"]
patients_hm = df_hm_fm_scaled["patient"]

X_bj = df_bj_fm_scaled[FINAL_COLS]
y_bj = df_bj_fm_scaled["Récidive Globale"]
patients_bj = df_bj_fm_scaled["patient"]

In [ ]:
results_hm_fm = pd.DataFrame(
    columns=[
        "Model",
        "Accuracy",
        "F1-Score",
        "PPV",
        "NPV",
        "Sensitivity",
        "Specificity"
    ]
)

results_bj_fm = pd.DataFrame(columns=results_hm_fm.columns)
results_hm_fm["Model"] = list(models.keys())
results_bj_fm["Model"] = list(models.keys())

cm_results_hm = {}
cm_results_bj = {}
cm_results = {}
dict_models = {}
for name, model in models.items():
    print(f"Training {name}...", end="")
    model.fit(X_train, y_train)  # Train the model

    y_pred_hm = model.predict_proba(X_hm)[:,1]  # Predict on HM
    accuracy, f1, PPV,NPV, Sensitivity,Specificity,fpr,tpr,roc_auc,  y_pred_hm,y_hm_2 = compute_metrics_fm(y_hm.to_list(), y_pred_hm,patients_hm)
    results_hm_fm.loc[results_hm_fm["Model"] == name] = [
        name,
        accuracy,
        f1,
        PPV,
        NPV,
        Sensitivity,
        Specificity
    ]
    cm_results_hm[name] = confusion_matrix(y_hm_2, y_pred_hm>0.5)
    precision, recall, thresholds = precision_recall_curve(y_hm_2, y_pred_hm)
    plt.figure()
    plt.plot(recall, precision, color='blue', lw=2)
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('Recall')
    plt.ylabel('Precision')
    plt.title('Precision-Recall curve on Henri Mondor')
    plt.show()
    y_pred_bj, y_bj_2
    print(" Done!")

    plt.figure()
    plt.plot(fpr, tpr, color='darkorange', lw=2, label='ROC curve (area = %0.2f)' % roc_auc)
    plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('Receiver Operating Characteristic on Henri Mondor')
    plt.legend(loc="lower right")
    plt.show()

    y_pred_bj = model.predict_proba(X_bj)[:,1]  # Predict on BJ
    accuracy, f1, PPV,NPV, Sensitivity,Specificity,fpr,tpr,roc_auc,  y_pred_bj, y_bj_2 = compute_metrics_fm(y_bj.to_list(), y_pred_bj,patients_bj)
    results_bj_fm.loc[results_bj_fm["Model"] == name] = [
        name,
        accuracy,
        f1,
        PPV,
        NPV,
        Sensitivity,
        Specificity
    ]
    cm_results_bj[name] = confusion_matrix(y_bj_2, y_pred_bj>0.5)
    plt.figure()
    plt.plot(fpr, tpr, color='darkorange', lw=2, label='ROC curve (area = %0.2f)' % roc_auc)
    plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('Receiver Operating Characteristic on Beaujon')
    plt.legend(loc="lower right")
    plt.show()

    
    precision, recall, thresholds = precision_recall_curve(y_bj_2, y_pred_bj)
    plt.figure()
    plt.plot(recall, precision, color='blue', lw=2)
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('Recall')
    plt.ylabel('Precision')
    plt.title('Precision-Recall curve on Beaujon')
    plt.show()
    y_pred_bj, y_bj_2
    print(" Done!")


In [ ]:


results_hm_fm = results_hm_fm.sort_values(by="Accuracy", ascending=False)
results_hm_fm["hopital"] = "HM"

results_bj_fm = results_bj_fm.sort_values(by="Accuracy", ascending=False)
results_bj_fm["hopital"] = "BJ"

ALL_res = pd.concat([results_hm_fm, results_bj_fm])
ALL_res

## Calibration analysis

In [ ]:

PROBA = np.concatenate(( y_pred_hm, y_pred_bj), axis=None)
PREDS = np.concatenate(( y_pred_hm>0.5, y_pred_bj>0.5), axis=None)
TRUE = np.concatenate((y_hm_2, y_bj_2))

bins = 15
fraction_of_positives, mean_predicted_value = calibration_curve(
    TRUE, PROBA, n_bins=bins, strategy="uniform"
)

fig, ax = plt.subplots(ncols=2, nrows=1, figsize=(10, 4), sharex=True)
ax[0].plot(mean_predicted_value, fraction_of_positives, "o-", label="Gaussian SVM")
ax[0].plot([0, 1], [0, 1], "--", color="gray", label="Perfect calibration")
ax[0].set_xlabel("Mean predicted probability")
ax[0].set_ylabel("Fraction of positives")
ax[0].set_title("Calibration Curve")
ax[0].legend()
ax[0].grid(True)

ax[1].hist(PROBA, bins=bins, range=(0, 1))
ax[1].set_xlabel("Predicted probability")
ax[1].set_ylabel("Count")
ax[1].set_title("Predicted Probability Distribution")
plt.tight_layout()
plt.savefig(f"Calibration Curve PB+HM+BJ (bins={bins}).jpg")
plt.show()